In [52]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
from tqdm import tqdm

In [53]:
# Configuracion
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cpu


In [54]:
# 1. DATOS DE EJEMPLO (comentarios en español)
comentarios = [
    # Positivos (1)
    "Me encantó este producto, es excelente",
    "Muy buen servicio, lo recomiendo totalmente",
    "La atención al cliente fue increíble",
    "Excelente calidad, superó mis expectativas",
    "Me siento muy satisfecho con mi compra",
    "El producto llegó antes de lo esperado",
    "Totalmente recomendado, funciona perfecto",
    "La mejor compra que he hecho este año",
    "El personal es muy amable y profesional",
    "Volvería a comprar sin dudarlo",
    
    # Negativos (0)
    "Pésimo servicio, no lo recomiendo",
    "El producto llegó roto y en mal estado",
    "Muy mala experiencia, no volveré a comprar",
    "La atención al cliente es terrible",
    "No funciona como esperaba, decepcionado",
    "El envío tardó mucho más de lo debido",
    "Producto de baja calidad, no vale lo que cuesta",
    "Me arrepiento de haber comprado esto",
    "El servicio al cliente no responde",
    "No cumple con lo prometido, muy malo"
]


In [55]:
etiquetas = [1] * 10 + [0] * 10  # 1 para positivos, 0 para negativos
etiquetas

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [56]:
# Crea dataFrame
df = pd.DataFrame({'comentarios': comentarios, 'sentimiento': etiquetas})
df

,comentarios,sentimiento
0,"Me encantó este producto, es excelente",1
1,"Muy buen servicio, lo recomiendo totalmente",1
2,La atención al cliente fue increíble,1
3,"Excelente calidad, superó mis expectativas",1
4,Me siento muy satisfecho con mi compra,1
5,El producto llegó antes de lo esperado,1
6,"Totalmente recomendado, funciona perfecto",1
7,La mejor compra que he hecho este año,1
8,El personal es muy amable y profesional,1
9,Volvería a comprar sin dudarlo,1


## DataSet personalizado para PyTorch

In [57]:
class SentimentDataset(Dataset):
    def __init__(self, comentarios, etiquetas, tokenizer, max_len=120):
        self.comentarios = comentarios
        self.etiquetas = etiquetas
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comentarios)

    def __getitem__(self, idx):
        texto = str(self.comentarios[idx])
        etiqueta = self.etiquetas[idx]

        # Tokenizar el texto
        encoding = self.tokenizer(
            texto,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(int(etiqueta), dtype=torch.long)
        }

In [58]:
# 3. Preparar datos
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [59]:
# Dividir datos en entrenamiento y puruebas
x_train, x_test, y_train, y_test = train_test_split(
    df['comentarios'].values,
    df['sentimiento'].values,
    test_size=0.3,
    random_state=23,
    stratify=df['sentimiento'].values
)

In [60]:
print(f'Tamaño del conjunto de entrenamiento: {len(x_train)}')
print(f'Tamaño del conjunto de prueba: {len(x_test)}')

Tamaño del conjunto de entrenamiento: 14
Tamaño del conjunto de prueba: 6


In [61]:
train_dataset = SentimentDataset(x_train, y_train, tokenizer)
test_dataset = SentimentDataset(x_test, y_test, tokenizer)
test_dataset


In [62]:
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)

## Entrenamiento del modelo

In [63]:
modelo = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
modelo = modelo.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [64]:
optimizador = AdamW(modelo.parameters(), lr=2e-5)

In [65]:
# Funcion de entrenamiento
def entrenar_modelo(modelo, train_dataloader, test_dataloader, optimizer, epocas=5):
    train_losses = []
    test_accuracies = []
    for epoca in range(epocas):
        print(f'Epoca {epoca + 1}/{epocas}: ')
        print('=' * 50)
        # Modo entrenamiento
        modelo.train()
        total_loss = 0
        
        for batch in tqdm(train_dataloader, desc='Entrenando'):
            # Mover datos al dispositivo
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            ## Pasar
            optimizer.zero_grad()
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss
            
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_dataloader)
        train_losses.append(avg_loss)
            
        # Evaluacion
        accuracy = evaluar_modelo(modelo, test_dataloader)
        test_accuracies.append(accuracy)
        print(f'Promedio de perdidas: {avg_loss:.4f} | Accuracy en prueba: {accuracy:.4f}')
            
    return train_losses, test_accuracies

In [66]:
def evaluar_modelo(modelo, test_dataloader):
    modelo.eval()
    predicciones = []
    verdaderos = []
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc='Evaluando'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = modelo(input_ids=input_ids, attention_mask=attention_mask)
            _, pred = torch.max(outputs.logits, dim=1)
            predicciones.extend(pred.cpu().numpy())
            verdaderos.extend(labels.cpu().numpy())
            
    accuracy = accuracy_score(verdaderos, predicciones)
    return accuracy
            

In [67]:
# Entranar el modelo
train_losses, test_accuracies = entrenar_modelo(
    modelo, train_dataloader, test_dataloader, optimizador, epocas=5
)

Epoca 1/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.20it/s]


Promedio de perdidas: 0.8235 | Accuracy en prueba: 0.5000
Epoca 2/5: 


Evaluando: 100%|██████████| 2/2 [00:00<00:00,  2.42it/s]


Promedio de perdidas: 0.7153 | Accuracy en prueba: 0.8333
Epoca 3/5: 


...

Promedio de perdidas: 0.6726 | Accuracy en prueba: 0.6667
Epoca 4/5: 


Evaluando: 100%|██████████| 2/2 [00:01<00:00,  1.58it/s]


Promedio de perdidas: 0.6922 | Accuracy en prueba: 0.3333
Epoca 5/5: 


Evaluando: 100%|██████████| 2/2 [00:01<00:00,  1.44it/s]

Promedio de perdidas: 0.7228 | Accuracy en prueba: 0.3333
